# 생성 파이프라인 — 코랩 실행본

**한 줄 이야기 -> 대본 -> 그림체 검색 -> 4컷 생성 -> 채점**이 끝까지 돕니다.
라이브 데모나 성능 튜닝을 이 위에서 하시면 됩니다.

```text
한 줄 이야기
 -> write_cuts      대본 4컷 (LLM 또는 목)
 -> retrieve_refs   그림체 검색 (Gram, daypack_v2)
 -> generate        4컷 생성 (gpt-image-1)
 -> evaluate        역방향 채점 + 천장 대비 정규화
```

## ★ 돈 안 쓰고 먼저 끝까지 돌려보세요

아래 3~7절은 **API 를 한 번도 안 부릅니다.** 파이프라인이 어떻게 도는지 먼저 보시고,
진짜 그림이 필요할 때만 8절로 가시면 됩니다 (4컷 약 $0.18).

## 준비물

| 무엇 | 어디서 |
|---|---|
| `daypack_v2.zip` | 팀 드라이브 (57MB). **내 드라이브에 바로가기 추가** 해두세요 |
| 코드 | 이 노트북이 공개 깃허브에서 받습니다. 따로 준비 안 하셔도 됩니다 |
| OpenAI 키 | **8절에서만** 필요합니다 |


## 1. GPU 확인

`런타임 -> 런타임 유형 변경 -> T4 GPU` 로 맞춰주세요. 없으면 임베딩이 매우 느립니다.


In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (T4 로 바꿔주세요)")


## 2. 드라이브 연결과 daypack_v2 찾기


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import glob, os

# 팀 폴더 이름이 두 가지로 쓰여서 둘 다 찾는다
FOLDER_NAMES = ["DLthon_RAG", "DLthon_그림체RAG"]
cands = []
for name in FOLDER_NAMES:
    cands += glob.glob(f"/content/drive/MyDrive/{name}")
    cands += glob.glob(f"/content/drive/MyDrive/*/{name}")
    cands += glob.glob(f"/content/drive/Shareddrives/*/{name}")
cands = [c for c in dict.fromkeys(cands) if glob.glob(f"{c}/**/daypack_v2.zip", recursive=True)]

if not cands:
    raise SystemExit(
        "daypack_v2.zip 이 있는 프로젝트 폴더를 못 찾았습니다.\n"
        "공유 문서함 폴더는 코랩에서 안 보입니다 — 드라이브에서 우클릭 -> "
        "'내 드라이브에 바로가기 추가' 후 이 셀을 다시 실행해 주세요.")
SHARE = cands[0]
ZIP = glob.glob(f"{SHARE}/**/daypack_v2.zip", recursive=True)[0]
print("프로젝트 폴더:", SHARE)
print("데이터:", ZIP)


## 3. 작업 폴더 만들기

`/content/work/` 아래에 코드와 데이터를 같은 구조로 놓습니다.
스크립트가 **`build/` 의 부모**를 프로젝트 뿌리로 잡기 때문에 이 구조를 지켜야 합니다.


In [ ]:
import os, zipfile, time

WORK = "/content/work"
for sub in ("build", "kit/encoders", "verify"):
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)          # daypack_v2/ 가 생긴다
# ★ 산출물은 드라이브에 떨어뜨립니다. /content 는 세션이 끝나면 통째로 날아갑니다.
E2E_OUT = f"{SHARE}/verify/e2e"
os.makedirs(E2E_OUT, exist_ok=True)
os.environ["E2E_OUT"] = E2E_OUT
print("산출물 저장 위치:", E2E_OUT)

PACK = f"{WORK}/daypack_v2"
assert os.path.isdir(PACK), f"{PACK} 이 안 생겼습니다. zip 안 구조를 확인해 주세요"

import csv
rows = list(csv.DictReader(open(f"{PACK}/meta.csv", encoding="utf-8")))
styles = sorted({r["style"] for r in rows})
print(f"그림 {len(rows)}장 / {len(styles)}클래스 ({time.time()-t0:.0f}초)")
assert len(rows) == 2522 and len(styles) == 20, "daypack_v2 구성이 예상과 다릅니다"


## 4. 코드 받기 — 공개 깃허브에서

**zip 을 따로 안 씁니다.** 깃허브가 단일 출처라, 여기서 받으면 항상 최신입니다.


In [ ]:
import urllib.error
import urllib.parse
import urllib.request

REPO_BASE = "https://raw.githubusercontent.com/MinWookGim/AIFFEL_Quest_eng/main/05_LLM/DLthon_RAG"
FILES = {
    "build/ab_prev_cut.py":  "05_생성_채점_실험/ab_prev_cut.py",
    "build/pipeline_e2e.py": "05_생성_채점_실험/pipeline_e2e.py",
    "build/cut1_guard.py":   "05_생성_채점_실험/cut1_guard.py",
    "build/style_ceiling.py":"05_생성_채점_실험/style_ceiling.py",
    "kit/encoders/clip_base.py":  "02_팀_공유/kit/encoders/clip_base.py",
    "kit/encoders/gram_vgg19.py": "02_팀_공유/kit/encoders/gram_vgg19.py",
    # 그림체별 천장 — 있으면 채점을 천장 대비로 읽는다 (없으면 절대값으로 읽어서 오해가 생긴다)
    "verify/style_ceiling_daypack_v2_gram.json":
        "05_생성_채점_실험/결과/style_ceiling_daypack_v2_gram.json",
}

missing = []
for dst, src in FILES.items():
    url = REPO_BASE + "/" + urllib.parse.quote(src)
    try:
        urllib.request.urlretrieve(url, f"{WORK}/{dst}")
        print(f"  받음  {dst}")
    except urllib.error.HTTPError as e:
        if e.code != 404:
            raise
        missing.append(src)
        print(f"  ★없음 {dst}  (404)")

if missing:
    raise SystemExit(
        "\n깃허브에서 못 찾은 파일이 있습니다:\n  "
        + "\n  ".join(missing)
        + "\n\n대개 원인은 하나입니다 — **저장소에 아직 안 올라간 것**입니다."
          "\n(코드 문제가 아닙니다. 저장소가 공개인지, 파일이 푸시됐는지 확인해 주세요.)"
          "\n확인: https://github.com/MinWookGim/AIFFEL_Quest_eng/tree/main/05_LLM/DLthon_RAG/05_%EC%83%9D%EC%84%B1_%EC%B1%84%EC%A0%90_%EC%8B%A4%ED%97%98")

print("\n코드 준비 끝")

# ★ 노트북은 코드와 달리 "한 번 받은 것"을 계속 쓴다. 옛 판이면 알려준다
NB_VERSION = "2026-08-10e"
try:
    latest = urllib.request.urlopen(
        REPO_BASE + "/" + urllib.parse.quote("05_생성_채점_실험/노트북_버전.txt"),
        timeout=10).read().decode().strip()
    if latest != NB_VERSION:
        print(f"\n★★ 이 노트북은 옛 판입니다 (내 판 {NB_VERSION} / 최신 {latest}).")
        print("   코드는 방금 최신으로 받았지만 **노트북의 절 구성은 옛 것**입니다.")
        print("   깃허브에서 DLthon_생성파이프라인_코랩.ipynb 를 다시 열어 주세요.")
    else:
        print(f"\n노트북 판본 {NB_VERSION} — 최신입니다")
except Exception:
    pass


In [ ]:
!pip -q install "openai>=1.40" 2>&1 | tail -1
print("준비 끝")


## 5. ★ 그림체 고르기

20개 중 무엇을 만들지 먼저 **눈으로 보고** 고릅니다. 아래 셀이 클래스마다 한 장씩 보여줍니다.


In [ ]:
import csv, collections, random
import matplotlib.pyplot as plt
from PIL import Image

rows = list(csv.DictReader(open(f"{PACK}/meta.csv", encoding="utf-8")))
by = collections.defaultdict(list)
for r in rows:
    by[r["style"]].append(r["file"])
names = sorted(by)

random.seed(0)
fig, axes = plt.subplots(4, 5, figsize=(13, 11))
for ax, name in zip(axes.ravel(), names):
    ax.imshow(Image.open(f"{PACK}/{random.choice(by[name])}").convert("RGB"))
    ax.set_title(f"{name}  (n={len(by[name])})", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("고를 수 있는 그림체:")
for i in range(0, len(names), 4):
    print("   " + "  ".join(f"{n:<26}" for n in names[i:i+4]))


아래 셀의 드롭다운에서 하나 고르세요. 이후 절이 이 값을 씁니다.

**천장이 낮은 그림체는 원래 어렵습니다** — `paint_Ukiyo_e` 0.125, `ink_m9` 0.221 처럼요.
처음에는 천장이 높은 쪽(`vec_undraw` 0.979, `vec_openmoji` 0.986)으로 감을 잡는 걸 권합니다.


In [ ]:
TARGET = "vec_undraw"  #@param ["ink_m1","ink_m2","ink_m3","ink_m4","ink_m5","ink_m6","ink_m7","ink_m8","ink_m9","paint_Art_Nouveau","paint_Baroque","paint_Cubism","paint_Impressionism","paint_Post_Impressionism","paint_Ukiyo_e","vec_irasutoya","vec_notoemoji","vec_openmoji","vec_twemoji","vec_undraw"]
STORY = "산길을 걷던 나그네가 노인을 만나 마을까지 함께 간다"  #@param {type:"string"}

import json
ceil = json.load(open(f"{WORK}/verify/style_ceiling_daypack_v2_gram.json", encoding="utf-8"))
c = {r["style"]: r["ceiling"] for r in ceil["rows"]}[TARGET]
print(f"목표 그림체: {TARGET}   천장 {c:.3f}")
print(f"이야기: {STORY}")
print("\n천장 = 진짜 그림을 넣었을 때 이 채점기가 주는 점수. 생성 점수는 이 값으로 나눠 읽습니다.")


## 6. 계획만 보기 (무료) — `--dry-run`

검색과 채점 눈금까지만 하고 **생성 직전에 멈춥니다.** 돈이 안 듭니다.
첫 실행은 2,522장 임베딩을 만드느라 40초쯤 걸리고, 그 다음부터는 캐시를 씁니다.


In [ ]:
%cd /content/work
!python build/pipeline_e2e.py --dry-run --target "$TARGET" --story "$STORY"


## 7. 끝까지 완주 (무료) — `--mock --no-llm`

생성은 자리표시 이미지, 대본도 목입니다. **파이프라인이 끝까지 도는지**만 봅니다.


In [ ]:
!python build/pipeline_e2e.py --mock --no-llm --target "$TARGET" --story "$STORY"


## 8. 나온 것 보기


In [ ]:
import glob, os
from IPython.display import Image as IPImage, display

# 비교 그림(레퍼런스 + 생성 4컷)이 있으면 그것부터. 한 장에 다 들어 있다
cmp_ = sorted(glob.glob(f"{E2E_OUT}/비교_*.png"), key=os.path.getmtime)
if cmp_:
    print("레퍼런스와 생성 컷을 나란히:")
    display(IPImage(cmp_[-1], width=1100))

cuts = sorted(glob.glob(f"{E2E_OUT}/e2e_*cut*.png"))
if cuts:
    print(f"\n낱장 {len(cuts)}개:")
    for p in cuts[:8]:
        print("  " + os.path.basename(p)); display(IPImage(p, width=300))


## 9. 진짜로 그림 뽑기 (여기서부터 돈이 듭니다)

**4컷 한 편에 약 $0.18.** 시간은 장당 **19~47초**, 한 편 **80~110초** 걸립니다(실측).
안전필터에 걸려 재시도가 붙으면 그 컷만 두 배쯤 더 걸립니다. **느린 게 정상입니다.**

★ **안전필터로 컷이 거부되는 일이 있습니다**(실측 16편 중 1편). 특히 `paint_Baroque` 처럼
서양 고전 회화가 레퍼런스로 들어가면 누드가 섞여 **출력 모더레이션(sexual)** 에 걸리기 쉽습니다.
한 컷이 막혀도 나머지로 이어가게 해뒀지만, 처음 돌려보실 때는 `--target vec_undraw` 를 권합니다.

키는 코랩 왼쪽 **열쇠 모양 `Secrets`** 에 `OPENAI_API_KEY` 로 넣고 Notebook access 를 켜주세요.
아래 셀은 키를 **출력하지 않습니다.**


In [ ]:
import os, getpass

try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if not key:
    key = getpass.getpass("OPENAI_API_KEY: ")

# 스크립트가 ~/.config/openai/api_key 를 읽는다. 코드를 안 고치고 파일로 넘긴다
os.makedirs(os.path.expanduser("~/.config/openai"), exist_ok=True)
p = os.path.expanduser("~/.config/openai/api_key")
with open(p, "w") as f:
    f.write(key)
os.chmod(p, 0o600)
del key
print("키 준비 완료 — 값은 표시하지 않습니다")

# 지출 기록을 드라이브에 둡니다. 코랩 세션이 끊겨도 누적이 이어집니다.
# (OpenAI 는 일반 API 키로 잔액을 알려주지 않습니다. 그래서 "내가 쓴 누적액"을 직접 적어둡니다.)
os.environ["SPEND_LOG"] = f"{SHARE}/verify/spend_log.json"
print("지출 기록:", os.environ["SPEND_LOG"])


### 얼마 썼는지 보기

`--budget` 을 주면 **누적 사용액과 남은 예산**을 같이 찍습니다.

```
    생성 합계 $0.196
    누적 사용 $0.245 (2회)  |  예산 $20.00 중 남음 $19.755
```

**잔액이 아니라 누적입니다.** OpenAI 가 일반 API 키로 잔액을 안 알려줍니다
(Usage API 는 Admin 키가 따로 필요하고, 그것도 사용량이지 잔액이 아닙니다).
금액은 응답의 usage 토큰 x 단가표로 낸 **추정치**이고, 실제 청구는 대시보드가 기준입니다.


In [ ]:
# 목표 그림체를 바꿔가며 돌려보세요. --sequential 을 붙이면 앞 컷을 물립니다
# --budget 은 이번 작업에 쓰기로 한 예산입니다 (남은 금액을 같이 보여줍니다)
!python build/pipeline_e2e.py --target "$TARGET" --story "$STORY" --budget 20


### 나온 그림 보기


In [ ]:
import glob, os
from IPython.display import Image as IPImage, display

cmp_ = sorted(glob.glob(f"{E2E_OUT}/비교_*.png"), key=os.path.getmtime)
if cmp_:
    display(IPImage(cmp_[-1], width=1200))
else:
    print("비교 그림이 없습니다. 위 셀이 끝까지 돌았는지 확인해 주세요.")


## 10. 채점을 무엇으로 하나 — 잣대 설명

점수를 믿으려면 **무엇을 어떻게 재는지**를 알아야 합니다. 짧게 적습니다.

### 어떻게 라벨을 붙이나 — 역방향 채점

생성한 그림을 **코퍼스에 질의로 던져** 가장 가까운 5장을 찾고, 그 5장의 **최다 라벨**을 그 컷의 판정으로 씁니다.
"이 그림은 무슨 그림체인가"를 분류기로 맞히는 게 아니라, **어디로 검색되는가**로 판정하는 것입니다.

★ 이때 **레퍼런스로 넣은 그림과 같은 원본은 후보에서 뺍니다.**
안 빼면 레퍼런스를 베낄수록 점수가 오릅니다 — 그림체가 아니라 복사를 재게 됩니다.

### 지표 넷

| 이름 | 무엇 | 어떻게 읽나 |
|---|---|---|
| `style_precision` | 4컷 중 목표 그림체로 판정된 비율 | **절대값으로 읽지 않습니다.** 아래 천장으로 나눠 읽습니다 |
| `style_precision_tail` | 컷2~4 만 | 컷1 은 조건마다 잡음이 커서 따로 봅니다 |
| `neighbor_hit` | 이웃 5장 중 목표 라벨 비율의 평균 | 라벨보다 부드러운 눈금 |
| `edition_cosine` | 4컷 서로의 코사인 평균(6쌍) | **일관성 판정에 쓰지 않습니다.** 아래 참고 |

### ★ 천장 — 왜 나눠서 읽나

**진짜 그림도 만점을 못 받습니다.** 코퍼스의 진짜 그림을 같은 방식으로 채점해 본 값이 "천장"입니다.

```
vec_openmoji 0.986   vec_undraw 0.979   paint_Baroque 0.713
ink_m3 0.657         paint_Impressionism 0.542   paint_Ukiyo_e 0.125
```

`paint_Ukiyo_e` 는 진짜 우키요에조차 0.125 입니다. 그러니 생성물이 0.25 를 받았다면
그건 낮은 게 아니라 **천장의 2배**입니다. 그래서 `천장 대비 %` 를 같이 찍습니다.

### 무작위로 찍으면 몇 점인가

daypack_v2 는 20클래스라 **무작위 기준선이 0.0516** 입니다 (클래스 크기를 반영한 값).
v1(9클래스)의 0.111 과 **다릅니다.** 두 숫자를 같은 표에 놓으면 안 됩니다.

### 컷 간 코사인을 일관성 판정에 안 쓰는 이유

실측에서 **판별에 실패**했습니다 — 진짜 그림 4장으로 재보니
**같은 그림체 0.668 vs 다른 그림체 0.653**, 차이가 0.015 라 분포가 겹칩니다.
게다가 8/9 실험에서 **코사인과 정확도의 상관이 0.290** 이었고, 코사인 상위 5편 중 3편은 정확도가 0 이었습니다.
**네 컷이 나란히 똑같이 틀린 것**입니다.

-> 그래서 코사인은 **복붙 탐지 상한(0.95)** 으로만 씁니다. 일관성은 **라벨 일치도**로 봅니다.

### 이 채점기의 알려진 약점 (숨기지 않고 적습니다)

- **자리표시용 회색 빈 상자**를 넣으면 `vec_undraw` 로 판정되고 `style_precision 1.000`,
  천장의 **102%** 가 나옵니다. Gram 이 vec_undraw 를 사실상 **"납작하고 질감 없음"** 으로 잡습니다.
  -> **vec 계열 점수가 높다고 "잘 그렸다"가 아닙니다.**
- 컷 단위 라벨 정확도는 v1 코퍼스에서 **0.440** 이었습니다(56% 오판).
  **v2 에서는 아직 안 쟀습니다.** 그래서 이 채점기는 **선별기로는 쓰되 판정기로는 안 씁니다**
  (재생성 루프를 꺼둔 이유입니다).

## 11. ★ 지금 아는 것 — 병목은 검색이 아닙니다

레퍼런스를 **정확히 찾아줘도** 생성이 목표 그림체를 부분적으로만 따라옵니다.

| 목표 그림체 | 생성 36컷 중 도메인을 맞춘 컷 |
|---|---|
| paint_Baroque (유화) | **34 / 36** |
| vec_undraw (납작한 벡터) | **5 / 36** |
| ink_m3 (가는 펜선) | **3 / 36** |

생성 모델이 음영·질감이 있는 쪽으로 끌립니다. 목표가 그 반대편이면 못 갑니다.
**검색 점수를 더 올려도 이 구간은 안 고쳐집니다.**

## 만지면 좋은 자리

| 자리 | 지금 | 해볼 것 |
|---|---|---|
| **생성 프롬프트** | 레퍼런스만 물리고 "스타일 따르라" | ★ **1순위.** 목표 그림체를 **말로 묘사**해 같이 넣기 |
| 컷1 선별 | best-of-3 을 Gram 점수로 | N 키우기 / 후보가 전부 빗나갈 때 처리 |
| 컷 간 일관성 | 복붙 상한(코사인 0.95)만 | 절대 코사인은 판별 실패. **라벨 일치도**가 낫습니다 |
| 재생성 루프 | 꺼짐 | 지금 채점기로 켜면 안 됩니다(56% 오판). 선별기가 먼저 |

## 숫자를 인용하실 때

- 전부 **daypack_v2**(2,522장/20클래스, 무작위 0.0516) 기준입니다
- **v1**(1,259장/9클래스, 무작위 0.111) 숫자와 섞지 마세요. 무작위가 달라 비교가 안 됩니다
- 95% 구간이 0을 품으면 "올랐다"가 아니라 **"아직 모른다"** 입니다

ink(수묵화) 이미지 출처: AI-Hub 168 한국화 데이터 — 한국지능정보사회진흥원 사업결과물
